In [6]:
import os
from autogen.agentchat import UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor
from dotenv import load_dotenv
from openai import AzureOpenAI
import json
import pandas as pd
import numpy as np
load_dotenv()

azure_gpt4o = {
    "api_type": "azure",
    "model": os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    "api_key": os.getenv('OPENAI_API_KEY'),
    "base_url": os.getenv('AZURE_OPENAI_ENDPOINT'),
    "api_version": os.getenv('OPENAI_API_VERSION')
}

In [7]:
#print(os.getenv('OPENAI_API_VERSION'))
data_dir = os.path.join(os.getcwd(), 'call-data')
codebook_file = os.path.join(data_dir, 'codebook.json')
transcripts_file = os.path.join(data_dir, 'transcripts.json')
sample_output_file = os.path.join(data_dir, 'example_output.csv')
print(sample_output_file)

2024-10-01-preview
/Users/kaiyrbekovk2/Repositories/ai-based-survey/call-data/example_output.csv


In [8]:
def get_QA_and_codebook(codebook_file):
    qa_list = []
    with open(codebook_file, 'r') as file:
        codebook = json.load(file)
        question_count = 1
        for code, content in codebook.items():
            qa_list.append(f"Question {question_count}: {content['question']}\nAnswers: ")
            ans_list = []
            for ans, ans_id in content['answer_to_answer_id'].items():
                ans_list.append(f"{ans}")
            
            qa_list.append(', '.join(ans_list))
            qa_list.append("\n")
            question_count += 1
    return ''.join(qa_list), codebook
QA_details, codebook = get_QA_and_codebook(codebook_file)
print(QA_details)

Question 1: Generally speaking, would you say that you can trust all the people, most of the people, some of the people, or none of the people in your neighborhood?
Answers: All, Most, Some, None, DON'T KNOW, SKIPPED ON WEB, REFUSED
Question 2: In the past month, how often did you talk with any of your neighbors?
Answers: Basically every day, A few times a week, A few times a month, Once a month, Not at all, Not sure, SKIPPED ON WEB, REFUSED
Question 3: During a typical month prior to March 1, 2020, when COVID-19 began spreading in the United States, how often did you talk with any of your neighbors?
Answers: Basically every day, A few times a week, A few times a month, Once a month, Not at all, Not sure, SKIPPED ON WEB, REFUSED
Question 4: In the past month, how often did you communicate with friends and family by phone, text, email, app, or using the Internet?
Answers: Basically every day, A few times a week, A few times a month, Once a month, Not at all, Not sure, SKIPPED ON WEB, RE

In [14]:
client = AzureOpenAI(
  api_key = os.getenv('OPENAI_API_KEY'),  
  api_version = os.getenv('OPENAI_API_VERSION'),
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
)



role_description = f'''You are a helpful assistant that reads conversation transcipts and deduces answer given by each user to a question.
You return the answer for each question in order of they appear in the question list.
The returned answer needs to be a single line, with answers separated by commas and no other punctuation.
Here are the list of questions with respective answer options:
{QA_details}
'''
#print(role_description)



with open(transcripts_file, 'r') as file:
    transcripts = json.load(file)

task_prompt = f"""Help me to understand the following conversation transcript : 

{transcripts["0"]}"""
#print(task_prompt)

conversation=[{"role": "system", "content": role_description}]
userid_to_answers = {}

for user_id, transcript in transcripts.items():
    task_prompt = f"""Help me to understand the following conversation transcript : 
                    {transcripts[user_id].replace('user:', 'surveyee:')}"""
    conversation.append({"role": "user", "content": task_prompt})

    #print(task_prompt)
    response = client.chat.completions.create(
        model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
        messages=conversation
    )
    userid_to_answers[user_id] = response.choices[0].message.content
    print(userid_to_answers[user_id])
    conversation.pop()

Some, Once a month, Once a month, Basically every day, Basically every day, No, No, Good, Yes, Yes
Some, A few times a month, A few times a month, Basically every day, A few times a month, No, No, Very good, No, No
Some, A few times a month, A few times a month, Basically every day, Basically every day, No, Yes, Very good, No, Yes
Some, A few times a week, A few times a week, Basically every day, Basically every day, No, Yes, Good, No, No
Some, Basically every day, Basically every day, Basically every day, Basically every day, No, No, Good, Yes, Yes
Most, A few times a month, A few times a month, Basically every day, Basically every day, Yes, Yes, Very good, No, No
Most, Once a month, A few times a month, Basically every day, Basically every day, No, No, REFUSED, Yes, No
Some, A few times a month, A few times a month, Basically every day, A few times a week, No, No, Good, No, No
All, Once a month, Not at all, A few times a month, A few times a month, No, No, Excellent, No, No
Some, Onc

In [15]:
answers_as_list = []
for user_id in sorted(list(userid_to_answers.keys())):
    user_answers = [part.strip() for part in userid_to_answers[user_id].split(',')]
    answers_as_list.append(user_answers)

In [17]:
question_code_to_answers = {}
i = 0
print(codebook['SOC1']['answer_to_answer_id']['Some'])
for code, val in codebook.items():
    question_code_to_answers[code] = []
    for user_answers in answers_as_list:
        ans_text = user_answers[i]
        question_code_to_answers[code].append(codebook[code]['answer_to_answer_id'][ans_text])
    i += 1

3


In [18]:
df = pd.DataFrame(question_code_to_answers)

In [19]:
df.to_csv('deduced.csv')